In [1]:
import csv
import json
import re
import ast
from pathlib import Path


INPUT_CSV = Path("F1DriversDataset.csv")
OUTPUT_JSON = Path("public/data/drivers.json")


def make_id(name: str) -> str:
    """Convert driver name to a stable ID."""
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def to_int(value):
    if value is None or value == "":
        return 0
    return int(float(value))


def to_float(value):
    if value is None or value == "":
        return 0.0
    return float(value)


def to_bool(value):
    return str(value).strip().lower() == "true"


drivers = []

with INPUT_CSV.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        driver_name = row["Driver"].strip()
        driver_id = make_id(driver_name)

        # Parse seasons list like "[1962, 1963]"
        seasons = []
        try:
            seasons = ast.literal_eval(row["Seasons"])
        except Exception:
            pass

        driver = {
            "id": driver_id,

            "fullName": driver_name,

            "image": f"/images/drivers/{driver_id}.jpg",

            "nationality": row["Nationality"],

            # Unknown from this dataset
            "number": None,
            "abbreviation": "",
            "currentTeamId": None,

            # Activity
            "active": to_bool(row["Active"]),

            # Seasons
            "seasons": seasons,
            "yearsActive": to_int(row["Years_Active"]),

            # Career stats
            "championships": to_int(row["Championships"]),
            "raceEntries": to_int(row["Race_Entries"]),
            "raceStarts": to_int(row["Race_Starts"]),
            "wins": to_int(row["Race_Wins"]),
            "podiums": to_int(row["Podiums"]),
            "polePositions": to_int(row["Pole_Positions"]),
            "fastestLaps": to_int(row["Fastest_Laps"]),
            "careerPoints": to_float(row["Points"]),

            # Calculated rates from dataset
            "winRate": to_float(row["Win_Rate"]),
            "podiumRate": to_float(row["Podium_Rate"]),
            "poleRate": to_float(row["Pole_Rate"]),
            "fastLapRate": to_float(row["FastLap_Rate"]),
            "pointsPerEntry": to_float(row["Points_Per_Entry"]),

            # Placeholder for live-season data
            "currentPoints": 0,
            "dnfs": 0,

            # Championship years
            "championshipYears": (
                row["Championship Years"].split(",")
                if row["Championship Years"]
                else []
            ),
        }

        drivers.append(driver)

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(drivers, f, indent=2, ensure_ascii=False)

print(f"Successfully wrote {len(drivers)} drivers to {OUTPUT_JSON}")

Successfully wrote 868 drivers to public\data\drivers.json


In [2]:
import csv
import json
import re
from pathlib import Path


def make_id(name: str) -> str:
    """
    Convert a circuit name into a stable identifier.
    Example:
        "Brands Hatch" -> "brands_hatch"
        "Circuit de Spa-Francorchamps" -> "circuit_de_spa_francorchamps"
    """
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


INPUT_CSV = Path("all_f1_circuits.csv")
OUTPUT_JSON = Path("public/data/tracks.json")

tracks = []

with INPUT_CSV.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        circuit_name = row["Circuit"].strip()
        track_id = make_id(circuit_name)

        track = {
            "id": track_id,
            "name": circuit_name,
            "city": row["City"].strip(),
            "country": row["Country"].strip(),
            "lengthKm": float(row["Track Length (km)"]),
            "turns": int(row["Turns"]),
            "direction": row["Direction"].strip(),
            "circuitType": row["Circuit Type"].strip(),
            "firstGrandPrix": row["First Grand Prix"].strip(),
            "lastGrandPrix": row["Last Grand Prix"].strip(),
            "racesHeld": int(row["Races"]),
            "lapRecord": {
                "time": row["Best Lap Timing"].strip(),
                "driver": row["Best Lap Driver"].strip(),
                "year": int(row["Best Lap Year"]),
                "milliseconds": float(row["Best Lap Time"]),
            },
            # Placeholder image path
            "layoutImage": f"/images/tracks/{track_id}.png",
            "active": True,
        }

        tracks.append(track)

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(tracks, f, indent=2, ensure_ascii=False)

print(f"Wrote {len(tracks)} tracks to {OUTPUT_JSON}")

Wrote 77 tracks to public\data\tracks.json
